# Eval: Tropical Cyclone (TC) From Predictions (Detailed)

This notebook unrolls the prediction-driven TC workflow in Python steps.

Important prerequisite:
- `metview` executable must be installed and on `PATH`.
  The module imports `DataRetriever`, which depends on Metview for GRIB reference loading.


In [ ]:
from pathlib import Path
import shutil
import json
import numpy as np
import matplotlib.pyplot as plt

HAVE_METVIEW = shutil.which('metview') is not None
print('metview in PATH:', HAVE_METVIEW)

if HAVE_METVIEW:
    from eval.tc.loading_predictions import discover_prediction_files
    from eval.tc.stats import extreme_tail_table
    from eval.tc.workflows import run_tc_pdf
else:
    discover_prediction_files = None
    extreme_tail_table = None
    run_tc_pdf = None
    print('Install/load metview first, then re-run this cell.')

In [ ]:
# Step 1: configure paths
predictions_dir = Path('/path/to/predictions_folder')   # contains predictions_YYYYMMDD_stepXXX.nc
outdir = Path('/tmp/tc_eval')
run_label = 'my_run'
base_tc_dir = '/home/ecm5702/hpcperm/data/tc'

outdir.mkdir(parents=True, exist_ok=True)


In [ ]:
# Step 2: discover and parse prediction files
if discover_prediction_files is not None:
    files = discover_prediction_files(predictions_dir)
    print('found files:', len(files))
    for item in files[:5]:
        print(item)

In [ ]:
# Step 3: extract prediction vectors inside the TC domain
# (This step requires loading prediction curves via the new API)
if discover_prediction_files is not None and predictions_dir.exists():
    from eval.tc.events import EVENTS
    from eval.tc.loading_predictions import select_prediction_files_for_event, load_prediction_curves
    files = discover_prediction_files(predictions_dir)
    event = EVENTS.get('idalia')
    if files and event:
        event_files = select_prediction_files_for_event(files, event)
        if event_files:
            curve = load_prediction_curves(event_files, bbox=event.bbox, support_mode='native')
            pred_msl = curve.msl
            pred_wind = curve.wind
            print('pred_msl size:', pred_msl.size)
            print('pred_wind size:', pred_wind.size)
        else:
            print('No predictions_*.nc files matched expected naming.')
    else:
        print('No prediction files or event not found.')

In [ ]:
# Step 4: inline histogram plots (prediction-only quick check)
if discover_prediction_files is not None and 'pred_msl' in locals():
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    axs[0].hist(pred_msl, bins=np.arange(980, 1021, 1), density=True, alpha=0.8, color='royalblue')
    axs[0].set_title('Prediction MSLP PDF')
    axs[0].set_xlabel('hPa')

    axs[1].hist(pred_wind, bins=np.arange(0, 35.01, 1), density=True, alpha=0.8, color='darkorange')
    axs[1].set_title('Prediction 10m wind PDF')
    axs[1].set_xlabel('m/s')

    for ax in axs:
        ax.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# Step 5: extreme-tail stats table (prediction curve only)
if extreme_tail_table is not None and 'pred_msl' in locals():
    tail = extreme_tail_table(
        {run_label: (pred_msl, pred_wind)},
        mslp_range=(980.0, 990.0),
        wind_threshold=25.0,
    )
    print(json.dumps(tail, indent=2))

In [ ]:
# Step 6: full reference comparison + PDF output (requires Metview + TC GRIB data)
RUN_FULL_TC = False

if run_tc_pdf is not None and RUN_FULL_TC:
    out_pdf = run_tc_pdf(
        outdir=str(outdir),
        prediction_dir=str(predictions_dir),
        run_label=run_label,
        grib_dir=base_tc_dir,
        extra_reference_expids=[],
    )
    print('Saved PDF:', out_pdf)
    stats_path = Path(out_pdf).with_suffix('.stats.json')
    print('Stats JSON:', stats_path)